# 🚀 FashionMNIST 修复版 - BasicBlock + 30 Epochs

## ❌ 之前的问题（B结果.md只有93%）

| 问题 | 原因 | 解决方案 |
|------|------|----------|
| **欠拟合**（Train 92% vs Test 93%） | Bottleneck残差块太激进，28x28小图信息丢失严重 | ✅ 改用BasicBlock（ResNet18风格） |
| 训练损失下不去（0.21） | Dropout太强（0.4+0.3） + 数据增强过强 | ✅ Dropout降到0.2，减弱增强 |
| 准确率卡在93%上不去 | Bottleneck的1x1卷积压缩特征，小数据集不适合 | ✅ BasicBlock用两个3x3，感受野更大 |

## ✅ 本版本改进

1. 🔥 **BasicBlock替代Bottleneck** - 两个3x3卷积，适合小数据集
2. ⚡ **减少epoch到30轮** - 足够收敛，节省时间（原50轮）
3. 🎯 **降低Dropout强度** - 从0.4/0.3降到0.2/0.2，让模型能学到东西
4. 📸 **减弱数据增强** - 去掉RandomErasing（对28x28太破坏）
5. ⚡ **AdamW + CosineAnnealing** - 保留

---

**🎯 目标准确率：95%+（训练集96%+）**

预计训练时间：**30-40分钟**（MPS） / **1-1.5小时**（CPU）

---

**👉 依次运行下面的cells开始训练！**


In [ ]:
# ============================================
# 1. 导入必要的库
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

print('✅ 所有库导入成功！')

# 设置随机种子保证可重复性
torch.manual_seed(42)
np.random.seed(42)
print('✅ 随机种子已设置（42）')


## 🔥 核心改进：BasicBlock（ResNet18风格）

### 为什么用BasicBlock？

**之前的Bottleneck问题**：
```
Bottleneck: 1x1(降维) → 3x3 → 1x1(升维)
                     ↓
   28x28的小图 → 信息被过度压缩 → 学不到东西
```

**BasicBlock的优势**（ResNet18在ImageNet的成功证明）：
```
BasicBlock: 3x3 → 3x3（两个3x3卷积）
                     ↓
   感受野相当于5x5（第一层padding=1，第二层感受野扩展）
   不压缩通道数 → 信息完整保留
   参数更多但适合小数据集 → 能学到更多特征
```

### 结构对比

**Bottleneck**（你之前用的）：
```
输入 [batch, 64, 28, 28]
  │
  ├─ 1x1 conv: 64 → 16 (降维4倍)
  ├─ 3x3 conv: 16 → 16
  ├─ 1x1 conv: 16 → 64 (升维4倍)
  └─ + shortcut
```

**BasicBlock**（本版本用）：
```
输入 [batch, 64, 28, 28]
  │
  ├─ 3x3 conv: 64 → 64 (不降维！)
  ├─ BatchNorm + ReLU
  ├─ 3x3 conv: 64 → 64
  ├─ BatchNorm
  └─ + shortcut → ReLU
```

### 关键差异
- ✅ **无信息压缩**：通道数保持不变或翻倍
- ✅ **更大感受野**：两个3x3叠加 ≈ 5x5
- ✅ **更适合小数据**：FashionMNIST只有10类，需要更多特征
- ✅ **ResNet18同款**：ImageNet 70.5%准确率，证明有效

---

**下面定义BasicBlock类**


## 🔥 核心改进：BasicBlock（ResNet18风格）

### 为什么用BasicBlock？

**之前的Bottleneck问题**：
```
Bottleneck: 1x1(降维) → 3x3 → 1x1(升维)
                     ↓
   28x28的小图 → 信息被过度压缩 → 学不到东西
```

**BasicBlock的优势**（ResNet18在ImageNet的成功证明）：
```
BasicBlock: 3x3 → 3x3（两个3x3卷积）
                     ↓
   感受野相当于5x5（第一层padding=1，第二层感受野扩展）
   不压缩通道数 → 信息完整保留
   参数更多但适合小数据集 → 能学到更多特征
```

### 结构对比

**Bottleneck**（你之前用的）：
```
输入 [batch, 64, 28, 28]
  │
  ├─ 1x1 conv: 64 → 16 (降维4倍)
  ├─ 3x3 conv: 16 → 16
  ├─ 1x1 conv: 16 → 64 (升维4倍)
  └─ + shortcut
```

**BasicBlock**（本版本用）：
```
输入 [batch, 64, 28, 28]
  │
  ├─ 3x3 conv: 64 → 64 (不降维！)
  ├─ BatchNorm + ReLU
  ├─ 3x3 conv: 64 → 64
  ├─ BatchNorm
  └─ + shortcut → ReLU
```

### 关键差异
- ✅ **无信息压缩**：通道数保持不变或翻倍
- ✅ **更大感受野**：两个3x3叠加 ≈ 5x5
- ✅ **更适合小数据**：FashionMNIST只有10类，需要更多特征
- ✅ **ResNet18同款**：ImageNet 70.5%准确率，证明有效

---

**下面定义BasicBlock类**


In [ ]:
# ============================================
# 2. 定义BasicBlock（残差块 - ResNet18风格）
# ============================================

class BasicBlock(nn.Module):
    """
    ResNet18/34使用的BasicBlock
    
    结构：两个3x3卷积 + shortcut连接
    特点：不降维，保持通道数不变（或翻倍靠stride）
    适合：小数据集（如FashionMNIST）
    """
    expansion = 1  # 扩展因子（Bottleneck是4，这里1表示通道数不变）
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        # 【第一层卷积】
        # 如果stride=2，这里会下采样（28x28 → 14x14）
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3,
            stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 【第二层卷积】
        # stride=1保持尺寸
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3,
            stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 【Shortcut路径】
        # 如果维度不匹配（通道数变或尺寸变），用1x1卷积调整
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels,
                    kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        residual = self.shortcut(x)
        
        # 主路径：conv1 → BN → ReLU → conv2 → BN
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        # 残差连接
        out += residual
        return F.relu(out)


print('✅ BasicBlock 定义完成！')

# 测试BasicBlock
print('\n🔍 测试BasicBlock:')
block = BasicBlock(64, 128, stride=2)
x = torch.randn(4, 64, 28, 28)
y = block(x)
print(f'  输入: {x.shape} → 输出: {y.shape}')
print('  预期：通道64→128，尺寸28→14（下采样2倍）')


## 🏗️ 完整模型：ProCNN_Basic（BasicBlock版）

### 网络架构
```
输入: (1, 28, 28)  # FashionMNIST灰度图
  │
  ├─ Conv1: 3x3, 64通道, padding=1 → 28x28
  │    (Stem层，快速提取低级特征)
  │
  ├─ Layer1: 2×BasicBlock(64→64, stride=1) → 28x28
  │    （保持尺寸，提取中级特征）
  │
  ├─ Layer2: 2×BasicBlock(64→128, stride=2) → 14x14
  │    （下采样2倍，增加通道数）
  │
  ├─ Layer3: 2×BasicBlock(128→256, stride=2) → 7x7
  │    （再下采样2倍，通道数翻倍）
  │
  ├─ AdaptiveAvgPool2d((1,1)) → (256, 1, 1)
  │    （全局平均池化，自动适应）
  │
  └─ FC: 256 → 128 → 10
       Dropout(0.2) + Linear + ReLU + BN + Dropout(0.2) + Linear
```

### 关键参数
- **总参数量**：约 **300K**（原始只有115K，多了但值得）
- **层数**：1 + 2×3 = 7层（可训练层）
- **Dropout**：0.2 + 0.2（降低过拟合风险）
- **优化器**：AdamW(lr=1e-3, weight_decay=5e-4)
- **Scheduler**：CosineAnnealingLR(T_max=30)
- **Epochs**：**30轮**（不是50轮，更快收敛）

### 与原始对比
| 特性 | 原始ProCNN | 本版ProCNN_Basic |
|------|-----------|------------------|
| 残差连接 | ❌ 无 | ✅ BasicBlock |
| 参数量 | ~115K | ~300K（2.6倍） |
| Dropout | 0.3 | 0.2 + 0.2（更温和） |
| 池化 | MaxPool + flatten | AdaptiveAvgPool |
| Epochs | 20 | **30**（适度增加） |
| 目标准确率 | 93% | **95%+** |

---

**运行下面代码创建模型**


## 🏗️ 完整模型：ProCNN_Basic（BasicBlock版）

### 网络架构
```
输入: (1, 28, 28)  # FashionMNIST灰度图
  │
  ├─ Conv1: 3x3, 64通道, padding=1 → 28x28
  │    (Stem层，快速提取低级特征)
  │
  ├─ Layer1: 2×BasicBlock(64→64, stride=1) → 28x28
  │    （保持尺寸，提取中级特征）
  │
  ├─ Layer2: 2×BasicBlock(64→128, stride=2) → 14x14
  │    （下采样2倍，增加通道数）
  │
  ├─ Layer3: 2×BasicBlock(128→256, stride=2) → 7x7
  │    （再下采样2倍，通道数翻倍）
  │
  ├─ AdaptiveAvgPool2d((1,1)) → (256, 1, 1)
  │    （全局平均池化，自动适应）
  │
  └─ FC: 256 → 128 → 10
       Dropout(0.2) + Linear + ReLU + BN + Dropout(0.2) + Linear
```

### 关键参数
- **总参数量**：约 **300K**（原始只有115K，多了但值得）
- **层数**：1 + 2×3 = 7层（可训练层）
- **Dropout**：0.2 + 0.2（降低过拟合风险）
- **优化器**：AdamW(lr=1e-3, weight_decay=5e-4)
- **Scheduler**：CosineAnnealingLR(T_max=30)
- **Epochs**：**30轮**（不是50轮，更快收敛）

### 与原始对比
| 特性 | 原始ProCNN | 本版ProCNN_Basic |
|------|-----------|------------------|
| 残差连接 | ❌ 无 | ✅ BasicBlock |
| 参数量 | ~115K | ~300K（2.6倍） |
| Dropout | 0.3 | 0.2 + 0.2（更温和） |
| 池化 | MaxPool + flatten | AdaptiveAvgPool |
| Epochs | 20 | **30**（适度增加） |
| 目标准确率 | 93% | **95%+** |

---

**运行下面代码创建模型**


In [ ]:
# ============================================
# 3. 定义ProCNN_Basic模型（BasicBlock版）
# ============================================

class ProCNN_Basic(nn.Module):
    """
    使用BasicBlock的优化版CNN
    
    结构：Conv → BasicBlocks × 3 → AdaptiveAvgPool → FC
    目标：解决Bottleneck导致的欠拟合问题
    """
    def __init__(self, num_classes=10):
        super().__init__()
        
        # 【Stem】初始卷积层
        # 3x3卷积，64通道，padding保持28x28
        self.conv1 = nn.Conv2d(1, 64, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # 【Backbone】3个残差层，每层2个BasicBlock
        # Layer1: 28x28, 64通道（不降采样）
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        
        # Layer2: 28x28 → 14x14, 64→128通道
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        
        # Layer3: 14x14 → 7x7, 128→256通道
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        
        # 【Head】全局平均池化
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 【分类头】降低Dropout强度（0.2）
        self.fc = nn.Sequential(
            nn.Dropout(0.2),           # 第一层Dropout（温和）
            nn.Linear(256, 128),      # 256 → 128
            nn.ReLU(),
            nn.BatchNorm1d(128),     # FC后加BN
            nn.Dropout(0.2),          # 第二层Dropout
            nn.Linear(128, num_classes)  # 128 → 10
        )
        
        # He初始化
        self._initialize_weights()

    def _make_layer(self, in_channels, out_channels, blocks, stride=1):
        """
        构建残差层
        
        参数：
        - blocks: 该层几个BasicBlock（这里是2）
        - stride: 第一个block的步长（是否下采样）
        """
        layers = []
        # 第一个block：可能下采样
        layers.append(BasicBlock(in_channels, out_channels, stride))
        # 后续blocks：保持尺寸
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        """He初始化"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(
                    m.weight, mode='fan_out', nonlinearity='relu'
                )
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # 输入：x shape = [batch, 1, 28, 28]
        
        # Stem
        x = F.relu(self.bn1(self.conv1(x)))
        
        # Backbone
        x = self.layer1(x)  # 28x28, 64
        x = self.layer2(x)  # 14x14, 128
        x = self.layer3(x)  # 7x7, 256
        
        # Head
        x = self.avgpool(x)  # [batch, 256, 1, 1]
        x = x.view(x.size(0), -1)  # [batch, 256]
        x = self.fc(x)  # [batch, 10]
        return x


# 测试模型
model = ProCNN_Basic(num_classes=10)
test_input = torch.randn(4, 1, 28, 28)
test_output = model(test_input)
print(f'✅ 模型创建成功！')
print(f'   输入尺寸: {test_input.shape}')
print(f'   输出尺寸: {test_output.shape}')
num_params = sum(p.numel() for p in model.parameters())
print(f'   模型参数量: {num_params:,}')
print(f'   （比原始115K多了约{num_params/115000:.1f}倍，但性能更强）')


## 📸 数据增强策略（减弱版）

### 为什么减弱？
- FashionMNIST只有**28x28**小图
- 之前的`RandomErasing` + `RandomAffine` 太激进，破坏了关键特征
- **本版策略**：保持核心增强，去掉破坏性强的

### 增强清单
| 增强方法 | 参数 | 作用 |
|---------|------|------|
| `RandomHorizontalFlip` | p=0.5 | 水平镜像（衣服可能对称） |
| `RandomRotation` | 15度 | 轻微旋转 |
| `ColorJitter` | brightness=0.1, contrast=0.1 | 灰度图轻微扰动 |
| **RandomErasing** | **已移除** | ❌ 28x28太小，会擦掉关键像素 |
| **RandomAffine** | **已移除** | ❌ 平移缩放会扭曲小图 |

### 标准化参数
- Mean = 0.2860（FashionMNIST真实均值）
- Std = 0.3530（真实标准差）

---

**运行下面代码**


## 📸 数据增强策略（减弱版）

### 为什么减弱？
- FashionMNIST只有**28x28**小图
- 之前的`RandomErasing` + `RandomAffine` 太激进，破坏了关键特征
- **本版策略**：保持核心增强，去掉破坏性强的

### 增强清单
| 增强方法 | 参数 | 作用 |
|---------|------|------|
| `RandomHorizontalFlip` | p=0.5 | 水平镜像（衣服可能对称） |
| `RandomRotation` | 15度 | 轻微旋转 |
| `ColorJitter` | brightness=0.1, contrast=0.1 | 灰度图轻微扰动 |
| **RandomErasing** | **已移除** | ❌ 28x28太小，会擦掉关键像素 |
| **RandomAffine** | **已移除** | ❌ 平移缩放会扭曲小图 |

### 标准化参数
- Mean = 0.2860（FashionMNIST真实均值）
- Std = 0.3530（真实标准差）

---

**运行下面代码**


In [ ]:
# ============================================
# 4. 数据增强定义（减弱版）
# ============================================

def get_transforms():
    """
    训练和测试的transform
    
    修改：去掉RandomErasing和RandomAffine（对28x28太破坏）
    """
    # 【训练集】温和的数据增强
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),  # 水平翻转
        transforms.RandomRotation(15),            # ±15度旋转
        transforms.ColorJitter(
            brightness=0.1,   # 亮度±10%（原0.2）
            contrast=0.1      # 对比度±10%（原0.2）
        ),
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,)),  # 真实均值标准差
        # 移除了RandomErasing（28x28太小）
    ])

    # 【测试集】只标准化
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,))
    ])

    return train_transform, test_transform


print('✅ 数据增强定义完成（减弱版）！')
print('\n📋 训练集增强：')
print('  ✓ RandomHorizontalFlip(p=0.5)')
print('  ✓ RandomRotation(15)')
print('  ✓ ColorJitter(brightness=0.1, contrast=0.1)')
print('  ✗ RandomErasing（已移除）')
print('  ✗ RandomAffine（已移除）')


## 🎯 训练和评估函数

### 关键点
- `model.train()`：启用Dropout/BatchNorm的训练模式
- `model.eval()`：关闭Dropout/BatchNorm的随机性
- `torch.no_grad()`：禁用梯度计算（省内存、加速）
- `clip_grad_norm_`：梯度裁剪（防爆炸，保留）

---

**运行下面代码**


## 🎯 训练和评估函数

### 关键点
- `model.train()`：启用Dropout/BatchNorm的训练模式
- `model.eval()`：关闭Dropout/BatchNorm的随机性
- `torch.no_grad()`：禁用梯度计算（省内存、加速）
- `clip_grad_norm_`：梯度裁剪（防爆炸，保留）

---

**运行下面代码**


In [ ]:
# ============================================
# 5. 训练和评估函数
# ============================================

def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    correct = 0
    total_samples = 0

    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        batch_size = images.size(0)
        total_samples += batch_size

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        
        # 梯度裁剪（防爆炸）
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()

        total_loss += loss.item()
        pred = outputs.argmax(dim=1)
        correct += (pred == labels).sum().item()

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{correct/total_samples:.2%}'
        })

    return total_loss / len(loader), correct / total_samples


def test_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        pbar = tqdm(loader, desc='Testing', leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            batch_size = images.size(0)
            total_samples += batch_size

            outputs = model(images)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            pred = outputs.argmax(dim=1)
            correct += (pred == labels).sum().item()

            pbar.set_postfix({'acc': f'{correct/total_samples:.2%}'})

    return total_loss / len(loader), correct / total_samples


print('✅ 训练和评估函数定义完成！')


## ⏰ 早停机制（EarlyStopping）

- patience=8（30轮训练，容忍8轮不提升）
- 自动保存最佳模型

---

**运行下面代码**


## ⏰ 早停机制（EarlyStopping）

- patience=8（30轮训练，容忍8轮不提升）
- 自动保存最佳模型

---

**运行下面代码**


In [ ]:
# ============================================
# 6. 早停类
# ============================================

class EarlyStopping:
    """早停机制"""
    def __init__(self, patience=8, delta=0):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_acc = 0
        self.early_stop = False

    def __call__(self, val_acc):
        if val_acc > self.best_acc + self.delta:
            self.best_acc = val_acc
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


print('✅ EarlyStopping 定义完成！')


## 📊 训练曲线可视化

4个子图：
1. Loss曲线（训练/测试）
2. Accuracy曲线（训练/测试）
3. 学习率曲线（CosineAnnealing）
4. Train-Test Gap（过拟合判断）

---

**运行下面代码**


## 📊 训练曲线可视化

4个子图：
1. Loss曲线（训练/测试）
2. Accuracy曲线（训练/测试）
3. 学习率曲线（CosineAnnealing）
4. Train-Test Gap（过拟合判断）

---

**运行下面代码**


In [ ]:
# ============================================
# 7. 可视化函数
# ============================================

def plot_history(history):
    """绘制训练曲线"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('训练曲线 (Training Curves)', fontsize=16)

    # Loss曲线
    axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0, 0].plot(history['test_loss'], label='Test Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('损失曲线')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Accuracy曲线
    axes[0, 1].plot([acc*100 for acc in history['train_acc']], 
                    label='Train Acc', linewidth=2)
    axes[0, 1].plot([acc*100 for acc in history['test_acc']], 
                    label='Test Acc', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].set_title('准确率曲线')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_ylim([0, 100])

    # 学习率曲线
    axes[1, 0].plot(history['lr'])
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Learning Rate')
    axes[1, 0].set_title('学习率调度（CosineAnnealing）')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_yscale('log')

    # Gap曲线
    gap = [train - test for train, test in zip(history['train_acc'], history['test_acc'])]
    axes[1, 1].plot([g*100 for g in gap], linewidth=2, color='red')
    axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1, 1].axhline(y=0.03, color='orange', linestyle='--', alpha=0.5, label='3%警戒线')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Gap (%)')
    axes[1, 1].set_title('训练-测试差距')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    print('📊 训练曲线已保存为 "training_history.png"')
    plt.show()

print('✅ 可视化函数定义完成！')


## 🚀 主训练流程

### 关键参数
- **epochs = 30**（不是50！更快收敛）
- **AdamW** + weight_decay=5e-4
- **CosineAnnealingLR** + T_max=30
- **batch_size = 128**
- **早停 patience=8**

### 预期时间
- MPS (Mac GPU): **30-40分钟**
- CUDA (NVIDIA): **20-30分钟**
- CPU: **1-1.5小时**

### 预期结果
| Epoch | Train Acc | Test Acc（目标） |
|-------|-----------|----------------|
| 10 | 90-92% | 91-93% |
| 20 | 94-95% | 93-94% |
| 30 | 96-97% | **95-96%** ✅ |

---

**⚠️  检查清单**
- [ ] 所有cells已按顺序运行
- [ ] 数据集会自动下载到data/文件夹
- [ ] 确保有足够磁盘空间（约50MB）
- [ ] 推荐用GPU（MPS/CUDA）

**运行下面代码开始训练！** ⬇️


## 🚀 主训练流程

### 关键参数
- **epochs = 30**（不是50！更快收敛）
- **AdamW** + weight_decay=5e-4
- **CosineAnnealingLR** + T_max=30
- **batch_size = 128**
- **早停 patience=8**

### 预期时间
- MPS (Mac GPU): **30-40分钟**
- CUDA (NVIDIA): **20-30分钟**
- CPU: **1-1.5小时**

### 预期结果
| Epoch | Train Acc | Test Acc（目标） |
|-------|-----------|----------------|
| 10 | 90-92% | 91-93% |
| 20 | 94-95% | 93-94% |
| 30 | 96-97% | **95-96%** ✅ |

---

**⚠️  检查清单**
- [ ] 所有cells已按顺序运行
- [ ] 数据集会自动下载到data/文件夹
- [ ] 确保有足够磁盘空间（约50MB）
- [ ] 推荐用GPU（MPS/CUDA）

**运行下面代码开始训练！** ⬇️


In [ ]:
# ============================================
# 8. 主训练流程
# ============================================

def main():
    """完整训练流程（30 epochs）"""
    
    # ════════════════════════════════════════
    # 1. 设备选择
    # ════════════════════════════════════════
    device = torch.device("mps" if torch.backends.mps.is_available() 
                         else "cuda" if torch.cuda.is_available() 
                         else "cpu")
    print(f"🖥️  使用设备: {device}")
    if device.type == 'mps':
        print('   ⚡ 使用Apple Silicon GPU (MPS)')
    elif device.type == 'cuda':
        print(f'   ⚡ 使用NVIDIA GPU: {torch.cuda.get_device_name(0)}')
    else:
        print('   ⚠️  使用CPU（建议用GPU，否则很慢）')

    # ════════════════════════════════════════
    # 2. 数据加载
    # ════════════════════════════════════════
    print('\n📥 加载数据集...')
    train_transform, test_transform = get_transforms()
    
    train_data = datasets.FashionMNIST(
        root="data", train=True, download=True, transform=train_transform
    )
    test_data = datasets.FashionMNIST(
        root="data", train=False, download=True, transform=test_transform
    )
    
    train_loader = DataLoader(
        train_data, batch_size=128, shuffle=True, 
        num_workers=2
    )
    test_loader = DataLoader(
        test_data, batch_size=128, shuffle=False,
        num_workers=2
    )
    print(f'  ✓ 训练集: {len(train_data)} 张')
    print(f'  ✓ 测试集: {len(test_data)} 张')

    # ════════════════════════════════════════
    # 3. 模型创建
    # ════════════════════════════════════════
    print('\n🏗️  创建模型...')
    model = ProCNN_Basic(num_classes=10).to(device)
    num_params = sum(p.numel() for p in model.parameters())
    print(f'  ✓ 模型参数量: {num_params:,}')

    # ════════════════════════════════════════
    # 4. 损失 + 优化器 + Scheduler
    # ════════════════════════════════════════
    print('\n⚙️  配置训练参数...')
    loss_fn = nn.CrossEntropyLoss()
    
    # AdamW + weight_decay
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=1e-3,
        weight_decay=5e-4
    )
    
    # CosineAnnealingLR（30轮一个周期）
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=30, eta_min=1e-5
    )

    # 早停（30轮训练，容忍8轮）
    early_stopping = EarlyStopping(patience=8)

    # ════════════════════════════════════════
    # 5. 训练循环
    # ════════════════════════════════════════
    epochs = 30  # ⭐ 从50降到30，更快
    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': [],
        'lr': []
    }

    print('\n' + '='*60)
    print('🎯 开始训练！目标：95%+ (30 epochs)')
    print('='*60)

    best_acc = 0
    best_epoch = 0

    for epoch in range(epochs):
        print(f'\nEpoch {epoch+1:3d}/{epochs}')
        print('-' * 40)

        # 训练
        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, loss_fn, device
        )

        # 测试
        test_loss, test_acc = test_epoch(
            model, test_loader, loss_fn, device
        )

        # 更新学习率
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        # 记录
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['lr'].append(current_lr)

        # 打印
        print(f'\n📈 Epoch {epoch+1} 结果:')
        print(f'   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%}')
        print(f'   Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.2%}')
        print(f'   LR: {current_lr:.6f}')

        # 保存最佳模型
        if test_acc > best_acc:
            best_acc = test_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'best_procnn_basic.pth')
            print(f'  💾 保存最佳模型！(Acc: {best_acc:.2%})')

        # 早停检查
        early_stopping(test_acc)
        if early_stopping.early_stop:
            print(f'\n⚠️  早停触发！最佳epoch={best_epoch}, 最佳acc={early_stopping.best_acc:.2%}')
            break

    # ════════════════════════════════════════
    # 6. 完成
    # ════════════════════════════════════════
    print('\n' + '='*60)
    print(f'✅ 训练完成！')
    print(f'   最佳测试准确率: {best_acc:.2%} (epoch {best_epoch})')
    print(f'   总训练轮数: {epoch+1}/{epochs}')
    print('='*60)

    # ════════════════════════════════════════
    # 7. 可视化
    # ════════════════════════════════════════
    print('\n📊 正在生成训练曲线...')
    plot_history(history)

    # ════════════════════════════════════════
    # 8. 最终评估
    # ════════════════════════════════════════
    print('\n🔍 加载最佳模型进行最终评估...')
    model.load_state_dict(torch.load('best_procnn_basic.pth'))
    final_loss, final_acc = test_epoch(model, test_loader, loss_fn, device)
    print(f'\n🏆 最终测试准确率: {final_acc:.2%}')
    
    if final_acc >= 0.95:
        print('✨ ✨ ✨ 恭喜！达到95%+目标！ ✨ ✨ ✨')
    else:
        print('⚠️  未达95%目标，可以尝试：')
        print('   1. 增加epoch到40-50')
        print('   2. 使用数据增强（当前已减弱）')
        print('   3. 尝试更大模型（更多通道）')


# ============================================
# 9. 运行
# ============================================
if __name__ == "__main__":
    main()
